<a href="https://colab.research.google.com/github/joshlemonte/EarthDataViz/blob/main/RoseDiagram_Zion_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rose Diagram — Crossbed Dip Azimuths
## Zion NP Case Study  ·  GEOL 230 Earth Data Visualization  ·  BYU

---

### What this notebook does

This notebook reads your field measurements, computes basic **circular statistics**, and produces a publication-quality **rose diagram** (polar histogram) of crossbed dip azimuths measured in the Navajo Sandstone.

The finished figure is exported as **`Figure2_InsetPlot.png`** at 300 dpi — ready to place directly into your Illustrator Figure 2 layout.

### What you will submit
| File | Where it comes from |
|---|---|
| `Figure2_InsetPlot.png` | Exported by the last cell of this notebook |
| `Zion_FieldMeasurements.csv` | Your tidy field dataset (input to this notebook) |

---

### Workflow at a glance
1. **Upload your CSV** (or use the sample data provided) — *Cell 3*
2. **Run all cells** in order (`Runtime → Run all` in Colab)
3. **Check the statistics** printed by Cell 4
4. **View and export** the rose diagram in Cells 5–6

> 💡 **Tip:** Cells marked with 🔵 are where you enter or adjust your own data. All other cells can be run as-is.


---
## Cell 1 — Install & import libraries

Run this cell first. No changes needed.


In [ ]:
# Standard scientific Python stack — all available in Colab by default
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

print("Libraries loaded ✓")


---
## Cell 2 — Load your field data

### Option A — Upload your CSV (recommended)
If you have completed your `Zion_FieldMeasurements.csv`:

1. In **Colab**: click the 📁 Files icon in the left sidebar → drag your CSV in → update `CSV_FILE` below to match the filename.
2. In **Jupyter**: place the CSV in the same folder as this notebook.

Your CSV must have at least these two columns (exact names required):

| Column | Description | Example |
|---|---|---|
| `measurement_type` | type of measurement | `crossbed_azimuth` |
| `value` | numeric measurement value | `195` |

All other columns in the [required tidy format](https://GEOL230.byu.edu) are still expected for your dataset submission, but only these two are needed for this plot.

---

### Option B — Enter data manually (if your CSV isn't ready yet)

Uncomment the block near the bottom of this cell and type your azimuths directly as a Python list. You can switch to Option A later.

---

> ⚠️ **Important:** `MEASUREMENT_TYPE` below must exactly match the string in your `measurement_type` column. The default is `'crossbed_azimuth'`. Change it if you used a different label (e.g. `'joint_strike'`).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# STUDENT INPUT — update the two variables below
# ════════════════════════════════════════════════════════════════════

CSV_FILE         = "Zion_FieldMeasurements.csv"   # ← your filename here
MEASUREMENT_TYPE = "crossbed_azimuth"              # ← must match your CSV exactly

# ════════════════════════════════════════════════════════════════════
# Option A: load from CSV  (runs by default)
# ════════════════════════════════════════════════════════════════════
df = pd.read_csv(CSV_FILE)
az_series = (
    df.loc[df["measurement_type"].eq(MEASUREMENT_TYPE), "value"]
    .dropna()
    .astype(float)
)
az = az_series.to_numpy() % 360    # wrap to [0, 360)

# ════════════════════════════════════════════════════════════════════
# Option B: enter azimuths manually  (uncomment to use instead of Option A)
# Replace the numbers below with your own measurements.
# ════════════════════════════════════════════════════════════════════
# az = np.array([
#     195, 210, 188, 205, 220,   # ST02
#     178, 190, 215,             # ST03
#     202, 185, 197,             # ST04
# ]) % 360

# ────────────────────────────────────────────────────────────────────
print(f"Measurement type : {MEASUREMENT_TYPE}")
print(f"Azimuths loaded  : {len(az)}")
print(f"Values (°)       : {np.round(az, 1).tolist()}")

if len(az) < 6:
    print("\n⚠️  Warning: fewer than 6 measurements — consider collecting more data.")


---
## Cell 3 — Circular statistics

Standard (arithmetic) statistics like mean and standard deviation **do not work correctly for angular data** because angles wrap around at 360°.
For example, the average of 350° and 10° should be 0° (north) — not 180° (south).

We use **circular statistics** instead:

| Statistic | Symbol | Meaning |
|---|---|---|
| Circular mean | $\bar{\theta}$ | True average direction |
| Resultant length | $R$ | How tightly clustered the data are (0 = random, 1 = perfectly unimodal) |
| Circular standard deviation | $s$ | Spread around the mean direction |

Run this cell as-is — no changes needed.


In [ ]:
# ── Circular statistics (Fisher 1993) ─────────────────────────────
rad  = np.deg2rad(az)
C    = np.mean(np.cos(rad))
S    = np.mean(np.sin(rad))

mean_az   = np.rad2deg(np.arctan2(S, C)) % 360   # circular mean (0–360°)
resultant = np.sqrt(C**2 + S**2)                  # mean resultant length R
circ_std  = np.rad2deg(np.sqrt(-2 * np.log(resultant + 1e-12)))  # circular std dev

n = len(az)

print("── Circular Statistics ──────────────────────────────")
print(f"  n                     : {n} measurements")
print(f"  Range                 : {az.min():.1f}° – {az.max():.1f}°")
print(f"  Circular mean (θ̄)    : {mean_az:.1f}°")
print(f"  Mean resultant (R)    : {resultant:.3f}  (0 = random, 1 = unimodal)")
print(f"  Circular std dev (s)  : {circ_std:.1f}°")
print()

# Interpret R for students
if resultant >= 0.8:
    print("  → Strong preferred direction (R ≥ 0.80): data are tightly clustered.")
elif resultant >= 0.5:
    print("  → Moderate preferred direction (0.50 ≤ R < 0.80).")
else:
    print("  → Weak or no preferred direction (R < 0.50): data may be bimodal or scattered.")


---
## Cell 4 — Plot settings

You can adjust the two variables below to customise your figure.
Everything else runs automatically.

| Variable | Default | What it controls |
|---|---|---|
| `BIN_WIDTH` | `20` | Width of each rose petal in degrees. Try 10 or 30 to compare. |
| `FORMATION` | `'Navajo Sandstone'` | Appears in the figure title — update to match your unit. |


In [ ]:
# ════════════════════════════════════════════════════════════════════
# STUDENT INPUT — adjust if needed
# ════════════════════════════════════════════════════════════════════

BIN_WIDTH = 20                       # ← degrees per bin (must divide 360 evenly)
FORMATION = "Navajo Sandstone"       # ← formation name for the title

# ════════════════════════════════════════════════════════════════════
# Colour settings — leave as-is (or change hex codes to customise)
# ════════════════════════════════════════════════════════════════════
BAR_CMAP   = "Blues"        # matplotlib colormap for bars (try "Oranges" or "Greens")
MEAN_COLOR = "#C05C1E"      # colour of the mean-vector arrow

print(f"Bin width : {BIN_WIDTH}°  →  {360 // BIN_WIDTH} bins total")
print(f"Formation : {FORMATION}")


---
## Cell 5 — Build the rose diagram

This cell constructs the polar histogram.
No changes needed — run it after Cell 4.

### How the rose diagram is built
- Azimuths are sorted into `BIN_WIDTH`° bins around the compass.
- Each bin is drawn as a bar whose **length = count** of measurements in that bin.
- The **orange arrow** shows the circular mean direction (not the arithmetic mean).
- Bars are shaded from light to dark blue: darker = more measurements in that bin.


In [ ]:
# ── Bin the data ──────────────────────────────────────────────────
bin_edges  = np.deg2rad(np.arange(0, 360 + BIN_WIDTH, BIN_WIDTH))
theta_rad  = np.deg2rad(az)
counts, _  = np.histogram(theta_rad, bins=bin_edges)
widths     = np.diff(bin_edges)
max_count  = counts.max()

# Colour each bar by count (darker = more data)
cmap      = plt.get_cmap(BAR_CMAP)
bar_colors = [cmap(0.35 + 0.55 * v / max_count) for v in counts]

# ── Figure layout ─────────────────────────────────────────────────
fig = plt.figure(figsize=(5.5, 6.0), facecolor="white")
ax  = fig.add_axes([0.08, 0.16, 0.84, 0.76], polar=True)
ax.set_facecolor("#F8FAFB")

# ── Draw bars ─────────────────────────────────────────────────────
ax.bar(
    bin_edges[:-1], counts,
    width=widths, align="edge",
    color=bar_colors, edgecolor="#0D47A1", linewidth=0.6,
    zorder=3,
)

# ── Mean-vector arrow ─────────────────────────────────────────────
arrow_len = max_count * 0.85
mean_rad  = np.deg2rad(mean_az)
ax.annotate(
    "", xy=(mean_rad, arrow_len), xytext=(0, 0),
    arrowprops=dict(
        arrowstyle="->", color=MEAN_COLOR, lw=2.2, mutation_scale=14
    ),
    zorder=6,
)
ax.plot(mean_rad, arrow_len, "o", color=MEAN_COLOR, markersize=5, zorder=7)

# ── Axis formatting ───────────────────────────────────────────────
ax.set_theta_zero_location("N")   # North at top
ax.set_theta_direction(-1)        # Clockwise (geological convention)

ax.set_thetagrids(
    [0, 90, 180, 270], labels=["N", "E", "S", "W"],
    fontsize=11, fontweight="bold", color="#1A2733",
)

r_ticks = np.arange(1, max_count + 1, max(1, int(np.ceil(max_count / 4))))
ax.set_yticks(r_ticks)
ax.set_yticklabels([str(int(r)) for r in r_ticks], fontsize=7.5, color="#90A4AE")
ax.set_rlabel_position(67.5)

ax.yaxis.grid(True, linestyle=":", linewidth=0.6, color="#90A4AE", alpha=0.7)
ax.xaxis.grid(True, linestyle="-",  linewidth=0.35, color="#90A4AE", alpha=0.45)
ax.spines["polar"].set_color("#90A4AE")

# ── Title ─────────────────────────────────────────────────────────
ax.set_title(
    f"Cross-bed dip azimuths\n{FORMATION}  (n = {n})",
    fontsize=11, fontweight="bold", color="#1A2733", pad=14,
)

# ── Stats text box ────────────────────────────────────────────────
stats_str = (
    f"Mean: {mean_az:.0f}°    "
    f"R = {resultant:.2f}    "
    f"Circ. std: {circ_std:.0f}°    "
    f"Bin: {BIN_WIDTH}°"
)
fig.text(
    0.50, 0.058, stats_str,
    ha="center", va="center", fontsize=8, color="#1A2733",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#EEF2F7",
              edgecolor="#90A4AE", linewidth=0.8),
)

# ── Mean-vector legend ────────────────────────────────────────────
fig.text(
    0.50, 0.022,
    f"▶  Mean vector = {mean_az:.0f}°   ·   Stations: ST02, ST03, ST04, ST06",
    ha="center", va="center", fontsize=7.5, color=MEAN_COLOR, style="italic",
)

plt.show()
print("\nFigure preview shown above.")


---
## Cell 6 — Export at 300 dpi

Saves `Figure2_InsetPlot.png` to your current working directory.

- In **Colab**: after running, click the 📁 Files icon → right-click the file → *Download*.
- In **Jupyter**: the file will appear in the same folder as this notebook.

> This is the file you submit as `Figure2_InsetPlot.png` and place in your Illustrator layout.


In [ ]:
OUT_FILE = "Figure2_InsetPlot.png"
DPI      = 300

fig.savefig(OUT_FILE, dpi=DPI, bbox_inches="tight", facecolor="white")
print(f"Saved → {OUT_FILE}  ({DPI} dpi)")
print(f"File size: {os.path.getsize(OUT_FILE) / 1024:.0f} KB")


---
## Cell 7 — How to interpret your rose diagram

Use the statistics printed in Cell 3 and the diagram above to write your **callout** and **inset caption** for Figure 2.

### Reading the resultant length (R)
| R value | Interpretation |
|---|---|
| R ≥ 0.80 | Strong unimodal clustering — one dominant transport direction |
| 0.50 ≤ R < 0.80 | Moderate clustering — preferred direction but some scatter |
| R < 0.50 | Weak or no preferred direction — possibly bimodal or variable winds |

### Translating mean azimuth to a wind direction
Crossbed **dip azimuths** record the direction sediment moved **(the downwind direction)**.
To report the paleowind direction, state the **opposite**:

> *"A mean dip azimuth of 198° (SSW) indicates sediment transport toward the southwest,
> consistent with northeasterly trade winds during Early Jurassic erg deposition
> (Marzolf, 1983; Blakey, 1994)."*

### Required callout elements (checklist)
- [ ] Measurement type + units (e.g., *crossbed dip azimuths, degrees*)
- [ ] Mean ± circular std. (e.g., *198° ± 18°*)
- [ ] n = count (e.g., *n = 16*)
- [ ] Station IDs (e.g., *ST02, ST03, ST04, ST06*)
- [ ] Unit / formation (*Navajo Sandstone*)
- [ ] One-sentence interpretation tied to geology

### Required inset caption (example)
> *Fig. 2 inset. Rose diagram of crossbed dip azimuths measured in the Navajo Sandstone
> (n = 16; mean = 198° ± 18° circ. std.). Data collected at Stations ST02–ST04 and ST06,
> 6–7 March 2026, Zion National Park, Utah.*

---
### References
- Fisher, N. I. (1993). *Statistical Analysis of Circular Data.* Cambridge University Press.
- Marzolf, J. E. (1983). Changing wind and hydrologic regimes during deposition of the Navajo and Aztec Sandstones. *SEPM Special Publication*, 31.
- Blakey, R. C. (1994). Paleogeographic and tectonic controls on some Lower and Middle Jurassic erg deposits. *SEPM Special Publication*, 57.
